In [1]:
%pip install pandas numpy scipy plotly
%pip install --upgrade jsonschema isoduration fastjsonschema
%pip install --upgrade nbformat ipykernel

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import iqr
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"

In [3]:
# Load the dataset
file_path = "data/processed/medicare_acs_analysis.csv"
df = pd.read_csv(file_path)

# Data validation function
def validate_data(df):
    validation_results = {}

    # 1. Check for missing values
    missing_values = df.isnull().sum()
    validation_results['missing_values'] = missing_values[missing_values > 0].to_dict()

    # 2. Confirm data types
    expected_dtypes = {
        'county': 'object',
        'state': 'object',
        'inpatient_rate': 'float64',
        'outpatient_rate': 'float64',
        'ed_rate': 'float64',
        'socioeconomic_factor_1': 'float64',  # Replace with actual column names
        'socioeconomic_factor_2': 'float64'  # Replace with actual column names
    }
    dtype_issues = {}
    for col, expected_dtype in expected_dtypes.items():
        if col in df.columns and df[col].dtype != expected_dtype:
            dtype_issues[col] = {'expected': expected_dtype, 'actual': str(df[col].dtype)}
    validation_results['dtype_issues'] = dtype_issues

    # 3. Ensure utilization rates are scaled per 1,000 beneficiaries
    utilization_columns = ['inpatient_rate', 'outpatient_rate', 'ed_rate']
    utilization_issues = {}
    for col in utilization_columns:
        if col in df.columns:
            if not ((df[col] >= 0) & (df[col] <= 1000)).all():
                utilization_issues[col] = {
                    'min': df[col].min(),
                    'max': df[col].max()
                }
    validation_results['utilization_issues'] = utilization_issues

    return validation_results

# Run validation
validation_results = validate_data(df)

# Print validation results
print("Validation Results:")
for key, value in validation_results.items():
    print(f"{key}: {value}")

Validation Results:
missing_values: {'avg_annual_om_beneficiaries': 1, 'avg_age': 1, 'female_pct': 3, 'male_pct': 3, 'inpatient_stays_per_1000': 4, 'outpatient_visits_per_1000': 2, 'er_visits_per_1000': 3, 'median_household_income': 1}
dtype_issues: {}
utilization_issues: {}


In [4]:
# Drop rows with missing values
df = df.dropna()

## Descriptive Analysis

To analyze how inpatient, outpatient, and emergency department utilization rates vary geographically, we compute summary statistics (i.e., mean, median, interquartile range, standard deviation) and generate county-level choropleth maps across all three care settings. Through this, we expect to learn the baseline distribution, dispersion, and spatial clustering of healthcare utilization per 1,000 beneficiaries across U.S. counties.

In [5]:
utilization_columns = ['inpatient_stays_per_1000', 'outpatient_visits_per_1000', 'er_visits_per_1000']

descriptive_stats = {}

for col in utilization_columns:
    descriptive_stats[col] = {
        'mean': df[col].mean(),
        'median': df[col].median(),
        'iqr': iqr(df[col]),
        'std_dev': df[col].std()
    }

# Print descriptive statistics
print("Descriptive Statistics:")
for col, stats in descriptive_stats.items():
    print(f"{col}: {stats}")

# Generate county-level choropleth maps
# Ensure the DataFrame has a 'fips' column for county-level mapping
if 'fips' not in df.columns:
    raise ValueError("The dataset must contain a 'fips' column with valid U.S. county FIPS codes.")

# Create choropleth maps for each care setting
for col in utilization_columns:
    fig = px.choropleth(
        df,
        geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
        locations='fips',
        color=col,
        color_continuous_scale="Viridis",
        scope="usa",
        title=f"County-Level Choropleth Map: {col.replace('_', ' ').title()}",
        labels={col: f"{col.replace('_', ' ').title()} (per 1,000 beneficiaries)"}
    )
    fig.update_geos(fitbounds="locations", visible=False)
    fig.show()

Descriptive Statistics:
inpatient_stays_per_1000: {'mean': np.float64(215.47971246132215), 'median': np.float64(215.25837351212343), 'iqr': np.float64(53.71852294597255), 'std_dev': np.float64(41.10591996171189)}
outpatient_visits_per_1000: {'mean': np.float64(5451.299406859644), 'median': np.float64(5122.8161585136895), 'iqr': np.float64(3133.910331311811), 'std_dev': np.float64(2206.735537406361)}
er_visits_per_1000: {'mean': np.float64(588.5121662634278), 'median': np.float64(589.2006744875562), 'iqr': np.float64(134.80066182709982), 'std_dev': np.float64(107.84880897944801)}
